# First divergence with forward hooks
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dataflow-arch-for-ai/dataflow-arch-for-ai/blob/main/demos/week5/First_Divergence_Demo_Colab.ipynb)

**Dataflow Architectures for AI, Week 5, Module 5: Comparison against a baseline**  
**Author:** Dr. Kaoutar El Maghraoui

The method from the lecture: run the target and a baseline you trust on identical input, compare their outputs at layer boundaries, and find the first boundary where they disagree. The fault entered after the last match and before the first mismatch. You do not need to see inside a kernel to do this.

Two choices decide whether the comparison means anything. The first is the **reference**. Comparing two FP32 runs directly mixes two rounding errors together, so Module 4 uses a third run in FP64 as the reference and asks whether the target's error is within a small multiple of the baseline's error. The second is the **oracle**. For a layout or padding bug, a tolerance check can hide a single wrong lane, so Module 4 poisons the padding with a value that cannot appear in a correct result.

In this notebook you record every layer of Qwen 2.5 0.5B with forward hooks, compare a GPU run with a CPU baseline against an FP64 reference, plant a fault in one layer and find it, then run a poisoned-padding probe.

**What you will do**
1. Record the output of every decoder layer with forward hooks.
2. Record an FP64 reference and an FP32 baseline on the CPU.
3. Compare an FP32 GPU run, layer by layer, with the FP64 criterion.
4. Plant a fault in one layer and find the first divergence.
5. Use a poisoned-padding probe on a masked operation.

**Before you start:** Runtime > Change runtime type > **T4 GPU**. The FP64 run uses about 4 GB of CPU memory. The notebook takes about three minutes.

## 1. Forward hooks
A forward hook runs after a module's `forward` and receives its output. Registering one on every decoder layer gives a record of the whole model, keyed by layer index, without editing the model's code.

All runs below use the same token IDs, the same model weights and eval mode. The lecture's step two, identical inputs, is what makes a difference between two runs point at a fault.

In [ ]:
import gc
import torch
from transformers import AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
torch.manual_seed(0)
ids = torch.randint(0, 50000, (1, 32))            # the same 32 tokens for every run
print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())


def record_layers(model, input_ids, fault=None):
    """Run the model once and return each decoder layer's output on the CPU in FP64.

    fault: optional (layer index, function) that changes that layer's output, to plant a bug.
    """
    outputs, handles = {}, []
    for i, layer in enumerate(model.model.layers):
        def hook(module, args, out, i=i):
            hidden = out[0] if isinstance(out, tuple) else out
            if fault is not None and fault[0] == i:
                hidden = fault[1](hidden)
                out = (hidden,) + tuple(out[1:]) if isinstance(out, tuple) else hidden
            outputs[i] = hidden.detach().to("cpu", torch.float64)
            return out                                   # returning a value replaces the output
        handles.append(layer.register_forward_hook(hook))
    try:
        with torch.inference_mode():
            model(input_ids=input_ids.to(next(model.parameters()).device))
    finally:
        for h in handles:
            h.remove()
    return outputs


def load(dtype, device):
    return AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=dtype).to(device).eval()

## 2. The reference and the baseline
- **Reference:** FP64 on the CPU. It is slow, and its own rounding error is about a billion times smaller than FP32's.
- **Baseline:** FP32 on the CPU, a run you trust. Its error against the reference measures normal FP32 rounding for this model and input.

Each model is deleted after its run to keep memory low.

In [ ]:
model = load(torch.float64, "cpu")
ref = record_layers(model, ids)
del model; gc.collect()

model = load(torch.float32, "cpu")
base = record_layers(model, ids)
del model; gc.collect()
print(f"Recorded {len(ref)} layers for the reference and the baseline")

## 3. The criterion
For each layer, compute the baseline's error and the target's error against the FP64 reference, as the largest absolute difference. The target passes a layer if

`target error <= K x baseline error + ATOL`

with `K = 5` and a small `ATOL` for layers where the baseline error is close to zero. This is the test PyTorch's own accuracy checker uses. The criterion is fixed before looking at the target, as the lecture's step one says.

In [ ]:
K, ATOL = 5.0, 1e-5


def compare(target, label):
    first_bad = None
    print(f"{label}")
    print(f"{'layer':>5} {'baseline err':>13} {'target err':>11} {'ratio':>7}  result")
    for i in sorted(ref):
        e_base = (base[i] - ref[i]).abs().max().item()
        e_tgt = (target[i] - ref[i]).abs().max().item()
        ok = e_tgt <= K * e_base + ATOL
        if not ok and first_bad is None:
            first_bad = i
        if i % 4 == 0 or not ok or i == len(ref) - 1:
            print(f"{i:5d} {e_base:13.2e} {e_tgt:11.2e} {e_tgt / max(e_base, 1e-30):7.1f}  {'pass' if ok else 'FAIL'}")
    print("first divergence:", "none" if first_bad is None else f"layer {first_bad}")
    return first_bad

## 4. The target: FP32 on the GPU
The GPU uses different kernels and a different summation order from the CPU, so its numbers differ from the baseline's. The question is whether they differ by more than normal FP32 rounding.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
gpu_model = load(torch.float32, device)
gpu = record_layers(gpu_model, ids)
compare(gpu, "FP32 on the GPU against the FP64 reference")

**Reading the output**
- The baseline error grows quickly over the first layers, to about 1e-3, and stays near that level. Rounding errors accumulate through the residual stream, and a few large activations make the absolute numbers large.
- The GPU's error is the same order as the CPU baseline's at every layer, so every layer passes. The two runs differ from each other, but both are within FP32 rounding of the true result. Comparing GPU to CPU directly, without the reference, could not tell you that.

## 5. Plant a fault and find it
A real fault might be a wrong kernel for one shape, a layout bug, or a bad fusion. The cell simulates one by adding a small error, 0.01, to one channel of layer 13's output. Every later layer inherits it.

In [ ]:
FAULT_LAYER = 13


def small_bug(hidden):
    hidden = hidden.clone()
    hidden[..., 7] += 0.01                     # one channel slightly wrong
    return hidden


faulty = record_layers(gpu_model, ids, fault=(FAULT_LAYER, small_bug))
found = compare(faulty, f"FP32 on the GPU with a fault planted in layer {FAULT_LAYER}")
print(f"\nThe fault was planted in layer {FAULT_LAYER}; the first failing layer is {found}.")

**Reading the output**
- Layers 0 to 12 pass. Layer 13 is the first to fail, and every later layer fails as well, because the error flows on through the model.
- Only the first mismatch is informative. Later mismatches inherit the bad value and tell you nothing new, as the lecture's six-node example shows.
- The next step would be to rerun layer 13 alone, with its recorded input, to get a small reproducer.

## 6. Poisoned-padding probe
Batches of sequences with different lengths are padded to the same length. A correct masked operation never reads the padding. A tolerance check is the wrong oracle for this kind of bug: if the padding holds zeros, reading it changes the result only slightly.

The probe from the lecture: fill the padding with a value that cannot appear in a correct result, here NaN, and check that it never reaches the output. Two versions of masked mean pooling are tested. The buggy one multiplies by the mask, and NaN times zero is still NaN.

In [ ]:
lengths = torch.tensor([5, 8])                          # two sequences, padded to 8 positions
hidden = torch.randn(2, 8, 16)
mask = (torch.arange(8)[None, :] < lengths[:, None]).float()


def mean_pool_correct(h, m):
    h = torch.where(m[..., None].bool(), h, torch.zeros_like(h))   # padding never enters the sum
    return h.sum(1) / m.sum(1, keepdim=True)


def mean_pool_buggy(h, m):
    return (h * m[..., None]).sum(1) / m.sum(1, keepdim=True)    # padding is multiplied by zero


for label, fill in (("padding = 0.0", 0.0), ("padding = NaN (poisoned)", float("nan"))):
    h = hidden.clone()
    h[mask == 0] = fill
    good, bad = mean_pool_correct(h, mask), mean_pool_buggy(h, mask)
    print(f"{label}")
    print(f"  allclose(correct, buggy): {torch.allclose(good, bad)}")
    print(f"  NaN in buggy output:      {torch.isnan(bad).any().item()}")

**Reading the output**
- With zeros in the padding, the two versions agree, and `allclose` passes. The bug is invisible.
- With NaN in the padding, the buggy version's output for the padded sequence is NaN, and the probe finds it. The correct version is unaffected.
- On Spyre, the lecture uses the FP16 bit pattern `0x7FFF` as the poison, because it survives multiplication by zero on the device. The idea is the same: choose a value the correct code can never produce.

## Try this
1. In section 5, make the fault 100 times smaller (`+= 1e-4`). Is it still found at layer 13? What does that say about the size of fault this criterion can detect?
2. Load the GPU model in FP16 and compare it with the same criterion. Which layer fails first, and is that a bug? What baseline would you use for an FP16 target?
3. In section 6, test a mean that divides by the padded length (8) instead of the real length. Does the poisoned probe catch it? What would?

## Summary
- Compare a target and a baseline at layer boundaries on identical inputs. The first boundary over the threshold bounds the fault.
- Use an FP64 reference and compare the target's error with the baseline's. Two FP32 runs always differ a little.
- Forward hooks record every layer's output without changing the model's code.
- For padding and layout bugs, poison the padding with a value that cannot appear, instead of trusting a tolerance.

Lecture: Week 5, Module 4, slides 36 and 39 (FP64 reference, poisoned-lane probes), and Module 5, slides 43 to 46 (four steps, first divergence, forward hooks, controlled runs).